# Experiment 4 - Comparative Study of Deep CNN Architectures Using Transfer Learning

CIFAR-10 image classification using ImageNet-pretrained MobileNetV2, transfer learning, fine-tuning, evaluation metrics, and the mandatory plots specified in the CS3807 laboratory manual.


In [ ]:
import os, time, json, random
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_recall_fscore_support

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

OUT = Path("outputs")
OUT.mkdir(exist_ok=True)
print("TensorFlow:", tf.__version__)
print("CPU devices:", tf.config.list_physical_devices('CPU'))

In [ ]:
# Load CIFAR-10
(x_all, y_all), (x_test, y_test) = keras.datasets.cifar10.load_data()
y_all = y_all.ravel()
y_test = y_test.ravel()

# Stratified 90/10 split of the original 50,000-image training set
x_train, x_val, y_train, y_val = train_test_split(
    x_all, y_all, test_size=5000, random_state=SEED, stratify=y_all
)

class_names = ['Airplane', 'Automobile', 'Bird', 'Cat', 'Deer', 'Dog', 'Frog', 'Horse', 'Ship', 'Truck']

print('Original training set:', x_all.shape, y_all.shape)
print('Training split:', x_train.shape, y_train.shape)
print('Validation split:', x_val.shape, y_val.shape)
print('Testing set:', x_test.shape, y_test.shape)
print('Pixel range:', x_all.min(), 'to', x_all.max())

In [ ]:
# Mandatory plot 1: one CIFAR-10 sample from each class
fig, axes = plt.subplots(2, 5, figsize=(10, 4.4))
for cls, ax in enumerate(axes.ravel()):
    idx = np.flatnonzero(y_all == cls)[0]
    ax.imshow(x_all[idx])
    ax.set_title(class_names[cls], fontsize=9)
    ax.axis('off')
fig.suptitle('CIFAR-10: One Sample from Each Class', fontsize=12)
plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.savefig(OUT/'sample_cifar10_images.png', dpi=220, bbox_inches='tight')
plt.show()

In [ ]:
# ImageNet-pretrained convolutional base
preprocess_input = keras.applications.mobilenet_v2.preprocess_input
base_model = keras.applications.MobileNetV2(
    weights='imagenet', include_top=False, input_shape=(32, 32, 3), pooling='avg'
)
base_model.trainable = False
print('Base parameters:', f'{base_model.count_params():,}')
print('Feature dimension:', base_model.output_shape)

In [ ]:
# Extract frozen ImageNet features once. This preserves transfer learning while making
# the hyperparameter comparisons and classifier training substantially faster.
def extract_features(x, name):
    t0 = time.perf_counter()
    x_p = preprocess_input(x.astype('float32'))
    f = base_model.predict(x_p, batch_size=256, verbose=1)
    elapsed = time.perf_counter() - t0
    print(f'{name}: {f.shape}, {elapsed:.2f} s')
    return f, elapsed

train_features, t_feat_train = extract_features(x_train, 'train features')
val_features, t_feat_val = extract_features(x_val, 'validation features')
test_features, t_feat_test = extract_features(x_test, 'test features')
feature_extraction_time = t_feat_train + t_feat_val + t_feat_test

In [ ]:
def make_head(units=128, optimizer='Adam', learning_rate=0.001):
    model = keras.Sequential([
        layers.Input(shape=(train_features.shape[1],)),
        layers.Dense(units, activation='relu', name='classifier_dense'),
        layers.Dropout(0.2, name='classifier_dropout'),
        layers.Dense(10, activation='softmax', name='classifier_output')
    ])
    if optimizer == 'Adam':
        opt = keras.optimizers.Adam(learning_rate=learning_rate)
    else:
        opt = keras.optimizers.SGD(learning_rate=learning_rate, momentum=0.9)
    model.compile(optimizer=opt, loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

# Task 3 settings from the manual
head = make_head(units=128, optimizer='Adam', learning_rate=0.001)
t0 = time.perf_counter()
history_head = head.fit(
    train_features, y_train,
    validation_data=(val_features, y_val),
    epochs=10, batch_size=32, verbose=2
)
head_training_time = time.perf_counter() - t0

before_loss, before_acc = head.evaluate(test_features, y_test, batch_size=256, verbose=0)
print(f'Before fine-tuning test accuracy: {before_acc:.4f}')
print(f'Head training time: {head_training_time:.2f} s')

In [ ]:
# Hyperparameter study requested in the manual.
# The baseline run above is reused as the first configuration.
study = [{
    'Learning Rate': 0.001, 'Batch Size': 32, 'Epochs': 10,
    'Optimizer': 'Adam', 'Dense Units': 128,
    'Validation Accuracy': float(max(history_head.history['val_accuracy']))
}]

configs = [
    dict(lr=0.0001, batch=32, epochs=10, optimizer='Adam', units=128),
    dict(lr=0.001, batch=64, epochs=20, optimizer='Adam', units=256),
    dict(lr=0.001, batch=32, epochs=10, optimizer='SGD', units=128),
]
for cfg in configs:
    m = make_head(cfg['units'], cfg['optimizer'], cfg['lr'])
    h = m.fit(
        train_features, y_train,
        validation_data=(val_features, y_val),
        epochs=cfg['epochs'], batch_size=cfg['batch'], verbose=0
    )
    study.append({
        'Learning Rate': cfg['lr'], 'Batch Size': cfg['batch'], 'Epochs': cfg['epochs'],
        'Optimizer': cfg['optimizer'], 'Dense Units': cfg['units'],
        'Validation Accuracy': float(max(h.history['val_accuracy']))
    })

study_df = pd.DataFrame(study).sort_values('Validation Accuracy', ascending=False).reset_index(drop=True)
study_df.to_csv(OUT/'hyperparameter_study.csv', index=False)
study_df

In [ ]:
# Rebuild a complete transfer-learning model and copy the trained classifier weights.
inputs = keras.Input(shape=(32, 32, 3), name='input_image')
x = preprocess_input(inputs)
x = base_model(x, training=False)
x = layers.Dense(128, activation='relu', name='classifier_dense')(x)
x = layers.Dropout(0.2, name='classifier_dropout')(x)
outputs = layers.Dense(10, activation='softmax', name='classifier_output')(x)
model = keras.Model(inputs, outputs, name='CIFAR10_MobileNetV2')

model.get_layer('classifier_dense').set_weights(head.get_layer('classifier_dense').get_weights())
model.get_layer('classifier_output').set_weights(head.get_layer('classifier_output').get_weights())

# Fine-tune the last part of MobileNetV2; BatchNorm layers remain frozen.
base_model.trainable = True
for layer in base_model.layers[:-20]:
    layer.trainable = False
for layer in base_model.layers[-20:]:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

trainable_params = int(sum(np.prod(v.shape) for v in model.trainable_weights))
total_params = int(model.count_params())
print('Total parameters:', f'{total_params:,}')
print('Trainable parameters during fine-tuning:', f'{trainable_params:,}')

t0 = time.perf_counter()
history_ft = model.fit(
    x_train, y_train,
    validation_data=(x_val, y_val),
    epochs=5, batch_size=32, verbose=2
)
fine_tune_time = time.perf_counter() - t0
print(f'Fine-tuning time: {fine_tune_time:.2f} s')

In [ ]:
# Mandatory accuracy and loss plots.
# Epochs 1-10: frozen-base classifier training. Epochs 11-15: partial fine-tuning.
epochs = np.arange(1, 16)
train_acc = history_head.history['accuracy'] + history_ft.history['accuracy']
val_acc = history_head.history['val_accuracy'] + history_ft.history['val_accuracy']
train_loss = history_head.history['loss'] + history_ft.history['loss']
val_loss = history_head.history['val_loss'] + history_ft.history['val_loss']

def save_curve(values, ylabel, title, filename):
    plt.figure(figsize=(7.5, 4.2))
    plt.plot(epochs, values, marker='o', markersize=3)
    plt.axvline(10.5, linestyle='--', linewidth=1, label='Fine-tuning begins')
    plt.xlabel('Epoch')
    plt.ylabel(ylabel)
    plt.title(title)
    plt.grid(alpha=0.25)
    plt.legend()
    plt.tight_layout()
    plt.savefig(OUT/filename, dpi=220, bbox_inches='tight')
    plt.show()

save_curve(train_acc, 'Accuracy', 'Training Accuracy vs Epoch', 'training_accuracy.png')
save_curve(val_acc, 'Accuracy', 'Validation Accuracy vs Epoch', 'validation_accuracy.png')
save_curve(train_loss, 'Categorical Cross-Entropy Loss', 'Training Loss vs Epoch', 'training_loss.png')
save_curve(val_loss, 'Categorical Cross-Entropy Loss', 'Validation Loss vs Epoch', 'validation_loss.png')

In [ ]:
# Final evaluation on the complete 10,000-image CIFAR-10 test set.
t0 = time.perf_counter()
final_loss, final_acc = model.evaluate(x_test, y_test, batch_size=256, verbose=1)
y_prob = model.predict(x_test, batch_size=256, verbose=1)
prediction_time = time.perf_counter() - t0
y_pred = np.argmax(y_prob, axis=1)

precision, recall, f1, _ = precision_recall_fscore_support(
    y_test, y_pred, average='weighted', zero_division=0
)
cm = confusion_matrix(y_test, y_pred)
report = classification_report(
    y_test, y_pred, target_names=class_names, output_dict=True, zero_division=0
)
report_df = pd.DataFrame(report).transpose()
report_df.to_csv(OUT/'classification_report.csv')

print(f'Test loss: {final_loss:.4f}')
print(f'Test accuracy: {final_acc:.4f}')
print(f'Weighted precision: {precision:.4f}')
print(f'Weighted recall: {recall:.4f}')
print(f'Weighted F1-score: {f1:.4f}')
print(f'Prediction/evaluation time: {prediction_time:.2f} s')

In [ ]:
# Mandatory plot 6: confusion matrix
fig, ax = plt.subplots(figsize=(8.0, 7.0))
im = ax.imshow(cm, cmap='Blues')
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
ax.set_xticks(range(10), labels=class_names, rotation=45, ha='right')
ax.set_yticks(range(10), labels=class_names)
ax.set_xlabel('Predicted Label')
ax.set_ylabel('True Label')
ax.set_title('MobileNetV2 CIFAR-10 Confusion Matrix')
threshold = cm.max() / 2
for i in range(10):
    for j in range(10):
        ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                fontsize=7, color='white' if cm[i, j] > threshold else 'black')
plt.tight_layout()
plt.savefig(OUT/'confusion_matrix.png', dpi=220, bbox_inches='tight')
plt.show()

In [ ]:
# Save all measured values used by the laboratory report.
metrics = {
    'seed': SEED,
    'model': 'MobileNetV2',
    'pretraining': 'ImageNet',
    'training_images': int(len(x_train)),
    'validation_images': int(len(x_val)),
    'testing_images': int(len(x_test)),
    'image_size': '32 x 32 x 3',
    'classes': 10,
    'optimizer': 'Adam',
    'initial_learning_rate': 0.001,
    'fine_tune_learning_rate': 1e-5,
    'batch_size': 32,
    'frozen_epochs': 10,
    'fine_tune_epochs': 5,
    'dense_units': 128,
    'before_fine_tune_test_accuracy': float(before_acc),
    'after_fine_tune_test_accuracy': float(final_acc),
    'test_loss': float(final_loss),
    'precision_weighted': float(precision),
    'recall_weighted': float(recall),
    'f1_weighted': float(f1),
    'feature_extraction_time_seconds': float(feature_extraction_time),
    'classifier_training_time_seconds': float(head_training_time),
    'fine_tuning_time_seconds': float(fine_tune_time),
    'training_time_seconds': float(head_training_time + fine_tune_time),
    'end_to_end_compute_time_seconds': float(feature_extraction_time + head_training_time + fine_tune_time),
    'prediction_time_seconds': float(prediction_time),
    'total_parameters': total_params,
    'trainable_parameters_fine_tuning': trainable_params,
    'training_accuracy_history': [float(v) for v in train_acc],
    'validation_accuracy_history': [float(v) for v in val_acc],
    'training_loss_history': [float(v) for v in train_loss],
    'validation_loss_history': [float(v) for v in val_loss],
    'confusion_matrix': cm.tolist(),
}
with open(OUT/'metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print(json.dumps({k: v for k, v in metrics.items() if not isinstance(v, list)}, indent=2))